# Parte 1 — Análise exploratória com APIs externas

Clima (Open-Meteo), feriados (Nager.Date), padrões geoespaciais e demanda do 1746 (2023–2024).

## Instalando bibliotecas necessárias

## Importando bibliotecas necessárias

In [ ]:
import basedosdados as bd
import pandas as pd
import requests

### Configurações

In [37]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Base de dados de chamados

### Baixando dados da base

In [39]:
df_chamados = bd.read_sql(
    "SELECT * FROM `datario.adm_central_atendimento_1746.chamado` WHERE data_particao >= '2023-01-01' AND data_particao <= '2024-12-31' LIMIT 1000",
    billing_project_id="desafio-pic",
)



Downloading: 100%|██████████|


### Análise de colunas 

In [47]:
df_chamados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id_chamado                        1000 non-null   object             
 1   id_origem_ocorrencia              1000 non-null   object             
 2   data_inicio                       1000 non-null   datetime64[us]     
 3   data_fim                          1000 non-null   datetime64[us]     
 4   id_bairro                         1000 non-null   object             
 5   id_territorialidade               1000 non-null   object             
 6   id_logradouro                     1000 non-null   object             
 7   numero_logradouro                 972 non-null    Int64              
 8   id_unidade_organizacional         1000 non-null   object             
 9   nome_unidade_organizacional       1000 non-null   object        

### Análise de dados duplicados
Nenhum registro duplicado foi encontrado

In [51]:
df_chamados.duplicated().sum()

np.int64(0)

### Análise de dados nulos
Foram encontradas as seguintes colunas com dados nulos:
- numero_logradouro: 28
- longitude: 423
- latitude: 28
- data_alvo_diagnostico: 1000
- data_real_diagnostico: 1000
- justificativa_status: 965

In [64]:
print(
    [
        (col, df_chamados[col].isna().sum())
        for col in df_chamados.columns[df_chamados.isna().any()].tolist()
    ]
)

[('numero_logradouro', np.int64(28)), ('longitude', np.int64(423)), ('latitude', np.int64(423)), ('data_alvo_diagnostico', np.int64(1000)), ('data_real_diagnostico', np.int64(1000)), ('justificativa_status', np.int64(965))]


### Análise descritiva dos dados

In [41]:
df_chamados.describe().T

,count,mean,min,25%,50%,75%,max,std
data_inicio,1000,2024-05-17 17:00:58.848000,2024-05-01 12:51:22,2024-05-09 17:02:16.500000,2024-05-20 13:41:10,2024-05-24 14:05:25,2024-05-31 19:52:12,NaN
data_fim,1000,2024-05-24 16:36:08.681999,2024-05-02 12:39:56,2024-05-14 15:44:32,2024-05-24 16:03:25.500000,2024-06-03 08:00:58.250000,2024-07-04 10:07:09,NaN
numero_logradouro,972.0,277.048354,0.0,38.0,103.0,326.5,3680.0,503.95839
longitude,577.0,-43.201524,-47.500442,-43.206911,-43.188838,-43.179153,-43.107261,0.180475
latitude,577.0,-22.888912,-22.988239,-22.924157,-22.910399,-22.90133,-4.943207,0.749171
data_alvo_finalizacao,1000,2024-05-31 09:05:46.320000,2024-05-10 00:00:00,2024-05-22 09:02:30,2024-05-30 02:36:00,2024-06-07 13:25:45,2024-07-11 11:26:00,NaN
data_alvo_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
data_real_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
tempo_prazo,1000.0,9.068,6.0,6.0,7.0,15.0,15.0,3.898535
reclamacoes,1000.0,0.012,0.0,0.0,0.0,0.0,1.0,0.10894


### Tratamento de dados nulos
Considerando as análises iniciais, considerei substituir os valores nulos de:
- colunas Int ou Float pela mediana
- colunas de latitude ou longitude pela moda
- colunas de datas não serão alteradas, pois todos os registros são nulos e até o momento não serão usados

#### Coluna numero_logradouro

In [69]:
df_chamados["numero_logradouro"] = df_chamados["numero_logradouro"].fillna(
    df_chamados["numero_logradouro"].median()
)
df_chamados["numero_logradouro"].isna().sum()

np.int64(0)

### Coluna longitude

In [ ]:
df_chamados["longitude"] = df_chamados["longitude"].fillna(
    df_chamados["longitude"].mode()[0]
)
df_chamados["longitude"].isna().sum()

np.int64(0)

#### Coluna latitude

In [70]:
df_chamados["latitude"] = df_chamados["latitude"].fillna(
    df_chamados["latitude"].mode()[0]
)
df_chamados["latitude"].isna().sum()

np.int64(0)

## Baixando dados da API Open-Meteo